In [1]:
# Standard library
import copy
import glob
import multiprocessing
import os
import time
import zipfile

# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# Related third party
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [3]:
def print_model_size(mdl):
    torch.save(mdl.state_dict(), "tmp.pt")
    print("%.2f MB" %(os.path.getsize("tmp.pt")/1e6))
    os.remove('tmp.pt')

In [4]:
input_size = (224,224)
mean = [0.485, 0.456, 0.406] 
std = [0.229, 0.224, 0.225]
transform = transforms.Compose([
    transforms.Resize(input_size),  # Resize to a fixed size
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [5]:
class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label, folder_name in enumerate(['Dog', 'Cat']):
            folder_path = os.path.join(self.root_dir, folder_name)
            for file_name in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file_name)
                
                try:
                    with Image.open(file_path) as img:
                        
                        if img.mode != 'RGB':
                            img = img.convert('RGB')
                        
                        if img.mode != 'RGB':
                            print(f"Skipping {file_path} because it does not have 3 channels (RGB)")
                            continue

                        self.image_paths.append(file_path)
                        self.labels.append(label)
                        
                except Exception as e:
                    print(f"Skipping {file_path} due to error: {e}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        
        with Image.open(image_path) as img:

            if img.mode != 'RGB':
                img = img.convert('RGB')

            if self.transform:
                img = self.transform(img)
            
        return img, label

In [6]:
dataset = CustomDataset(root_dir='../data/PetImages', transform=transform)

# Calculate split sizes
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

# Split dataset into train and test
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

Skipping ../data/PetImages\Dog\11702.jpg due to error: cannot identify image file '../data/PetImages\\Dog\\11702.jpg'


c:\FHDO\Research Thesis\project\embeddedNeuralNetwork\venv\lib\site-packages\PIL\TiffImagePlugin.py:864: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Skipping ../data/PetImages\Dog\Thumbs.db due to error: cannot identify image file '../data/PetImages\\Dog\\Thumbs.db'
Skipping ../data/PetImages\Cat\666.jpg due to error: cannot identify image file '../data/PetImages\\Cat\\666.jpg'
Skipping ../data/PetImages\Cat\Thumbs.db due to error: cannot identify image file '../data/PetImages\\Cat\\Thumbs.db'


In [7]:
# Create DataLoader for train and test sets
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
for images, labels in train_loader:
    print(images.shape)
    print(labels.shape)
    break

torch.Size([128, 3, 224, 224])
torch.Size([128])


In [8]:
def train_epoch(model, criterion, optimizer, data_loader, device,epoch):
    model.train()
    
    epoch_loss = 0.0
    num_batches = len(data_loader)
    
    for batch_idx, (image, target) in enumerate(tqdm(data_loader)):
        image, target = image.to(device), target.to(device)
        
        output = model(image)
        
        loss = criterion(output, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    print(f"Epoch = {epoch+1} || Training Loss: {avg_epoch_loss:.4f}")


def evaluate(model, criterion, data_loader, device,epoch):
    
    model.eval()
    
    epoch_loss = 0.0
    
    correct_predictions = 0
    total_predictions = 0
    
    num_batches = len(data_loader)
    
    with torch.no_grad():
       
        for image, target in tqdm(data_loader):
            image, target = image.to(device), target.to(device)
            output = model(image)
            loss = criterion(output, target)
            # Accumulate batch loss
            epoch_loss += loss.item()
            
            # Calculate accuracy
            _, predicted = torch.max(output, 1)  # Get the predicted class index
            correct_predictions += (predicted == target).sum().item()
            total_predictions += target.size(0)
            
    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    accuracy = correct_predictions / total_predictions
    
    print(f"Epoch = {epoch+1} || Test Loss: {avg_epoch_loss:.4f} || Test Accuracy: {accuracy:.4f}")

In [9]:
class MobileNet(torch.nn.Module):
    def __init__(self):
        super(MobileNet, self).__init__()
        self.model = models.mobilenet_v2(weights=None)  
        
        # for param in self.model.parameters():
        #     param.requires_grad = False
            
        
        
        self.model.classifier[1] = nn.Sequential(
            nn.Linear(in_features=self.model.classifier[1].in_features,out_features=512),
            nn.LeakyReLU(negative_slope=0.02,inplace=False),
            nn.BatchNorm1d(num_features=512),
            nn.Dropout(p=0.4,inplace=False),
            nn.Linear(in_features=512,out_features=2),
            nn.Softmax(dim=1))
        
        # print(self.model)

    def forward(self, x):
        x = self.model(x)
        return x

In [10]:
model = MobileNet()
model.load_state_dict(torch.load("../model/original_catndog_mobilenetv2.pth"))
print_model_size(model)

11.76 MB


In [11]:
# import platform
# chip = platform.processor()

# if chip == 'arm':
#     backend = 'qnnpack'
# elif chip in ['x86_64', 'i386']:
#     backend = 'fbgemm'
# else:
#     raise SystemError("Backend is not supported")

# print(f"Using {backend} backend engine for {chip} CPU")

backend = 'fbgemm'
torch.backends.quantized.engine = backend

In [12]:
# from torch.quantization.quantize_fx import prepare_fx, convert_fx,prepare_qat_fx
from torch.quantization.quantize_fx import prepare_qat_fx, prepare_fx, convert_fx
from torch.ao.quantization import QConfigMapping, get_default_qat_qconfig_mapping

example_inputs = (torch.randn(1, 3, 224, 224),)
# qconfig = {
#     "": torch.quantization.get_default_qat_qconfig(backend),
#     "module_name": {
#       #  "features.1.conv.1", None,    
#       #  "features.2.conv.0.0", None,
#       #  "features.2.conv.2", None,
#       #  "features.3.conv.1.0", None,
#       #  "features.3.conv.2", None,
#       #  "features.4.conv.1.0", None,
#       #  "features.4.conv.2", None,
#       #  "features.5.conv.1.0", None,
#       #  "features.7.conv.1.0", None,
#       #  "features.8.conv.1.0", None,
#       #  "features.9.conv.2", None,
#       #  "features.11.conv.1.0", None,
#       #  "features.12.conv.2", None,
#       #  "features.14.conv.1.0", None,
#       #  "features.14.conv.2", None,
#       #  "features.15.conv.1.0", None,
#       #  "features.16.conv.2", None,
#       #  "features.17.conv.0.0", None,
#       #  "features.17.conv.2", None,

#     }
# }
qconfig_mapping = (
    get_default_qat_qconfig_mapping(backend)
    # .set_module_name("features.1.conv.1", None)
    # .set_module_name("features.2.conv.0.0", None)
    # .set_module_name("features.2.conv.2", None)
)

model.train()
prepared_model = prepare_qat_fx(model, qconfig_mapping, example_inputs)

c:\FHDO\Research Thesis\project\embeddedNeuralNetwork\venv\lib\site-packages\torch\ao\quantization\observer.py:216: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  reduce_range will be deprecated in a future release of PyTorch."


In [13]:
# Train with QAT
num_epochs = 5
prepared_model = prepared_model.to(device)
criterion = nn.CrossEntropyLoss(reduction='mean')
optimizer = torch.optim.Adam(prepared_model.parameters(), lr = 0.0001)
for nepoch in range(num_epochs):
    train_epoch(prepared_model, criterion, optimizer, train_loader, device, nepoch)
    model_quantized = copy.deepcopy(prepared_model)
    model_quantized.to(torch.device("cpu"))
    model_quantized = convert_fx(model_quantized.eval())
    

    # Save the quantized model as a scripted fx model
    model_quantized.eval()
    scripted_model = torch.jit.trace(model_quantized, example_inputs)
    scripted_model.save(f"../model/Scriptedfx_int8_catndog_mobilenetv2_epoch{nepoch}.pt")

  0%|          | 0/157 [00:00<?, ?it/s]

Epoch = 1 || Training Loss: 0.3242


  0%|          | 0/157 [00:00<?, ?it/s]

Epoch = 2 || Training Loss: 0.3196


  0%|          | 0/157 [00:00<?, ?it/s]

Epoch = 3 || Training Loss: 0.3192


  0%|          | 0/157 [00:00<?, ?it/s]

Epoch = 4 || Training Loss: 0.3208


  0%|          | 0/157 [00:00<?, ?it/s]

Epoch = 5 || Training Loss: 0.3195


In [14]:
print("Evaluating quantized model...")
evaluate(model_quantized,criterion, test_loader,torch.device("cpu"),nepoch)
print_model_size(model_quantized)

Evaluating quantized model...


  0%|          | 0/40 [00:00<?, ?it/s]

Epoch = 5 || Test Loss: 0.3254 || Test Accuracy: 0.9908
3.30 MB


In [15]:
# Save quantized model but not converted
torch.save(model_quantized.state_dict(), "../model/notScripted_int8_catndog_mobilenetv2.pth")